# 04: BreastMNIST at batch size 64 (15 runs)

The paper states batch size 128 for MedMNIST2D, and notebooks 01 to 03 used 128. The authors'
released ResNet-50 (224) checkpoints for BreastMNIST
([Zenodo record 7782114](https://zenodo.org/records/7782114)) tell a different story: their
BatchNorm layers counted 297, 630 and 288 batches, all multiples of 9. With 546 training
images, 9 batches per epoch means a batch size of about 64. At 128 it would be 5 per epoch,
and the authors' other BreastMNIST checkpoints (ResNet-18 at 224, and both models at 28) do
show multiples of 5.

This notebook trains BreastMNIST 15 times at batch size 64, with everything else the same as
notebooks 02 and 03 (100 epochs, best-validation checkpoint, no seeds).

**Fixed in advance:** the dataset, the run count and every flag were set before the first run.
All 15 runs are reported, whatever the outcome.

Cell 7 compares the 15 batch-64 runs with the 15 batch-128 runs from notebooks 02 and 03 and
with the authors' three released runs (0.839, 0.906, 0.853), and saves `bs64_per_run.csv` and
`bs64_summary.csv`.

In [1]:
# Cell 1: confirm both GPUs are visible before doing anything else.
import torch

print("CUDA available:", torch.cuda.is_available())
n_gpus = torch.cuda.device_count()
print("GPU count:", n_gpus)
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}, "
          f"{round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)} GB VRAM")

if n_gpus < 1:
    raise RuntimeError("No GPU visible. Check NVIDIA drivers / CUDA install before continuing.")

CUDA available: True
GPU count: 2
  GPU 0: NVIDIA GeForce RTX 3090, 25.8 GB VRAM
  GPU 1: NVIDIA GeForce RTX 3090, 25.8 GB VRAM


In [2]:
# Cell 2: install the two packages this repo needs, if not already present.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "medmnist", "tensorboardX", "ACSConv"],
    check=True,
)

import medmnist
print("medmnist version:", medmnist.__version__)

medmnist version: 3.0.2


In [3]:
# Cell 3: get the authors' own code, unmodified, as a ZIP (no working `git`
# needed on this machine). Skips re-downloading if already present.
import os
import urllib.request
import zipfile

REPO_DIR = "experiments-main"

if not os.path.exists(REPO_DIR):
    print("Downloading experiments repo...")
    urllib.request.urlretrieve(
        "https://github.com/MedMNIST/experiments/archive/refs/heads/main.zip",
        "experiments.zip",
    )
    with zipfile.ZipFile("experiments.zip", "r") as z:
        z.extractall(".")
    print("Extracted to", REPO_DIR)
else:
    print(f"{REPO_DIR} already exists, skipping download.")

MEDMNIST2D_DIR = os.path.join(REPO_DIR, "MedMNIST2D")
MEDMNIST3D_DIR = os.path.join(REPO_DIR, "MedMNIST3D")
assert os.path.isdir(MEDMNIST2D_DIR) and os.path.isdir(MEDMNIST3D_DIR), \
    "Expected MedMNIST2D/ and MedMNIST3D/ folders inside the repo, check the extraction."
print("2D script:", os.path.join(MEDMNIST2D_DIR, "train_and_eval_pytorch.py"))
print("3D script:", os.path.join(MEDMNIST3D_DIR, "train_and_eval_pytorch.py"))

experiments-main already exists, skipping download.
2D script: experiments-main\MedMNIST2D\train_and_eval_pytorch.py
3D script: experiments-main\MedMNIST3D\train_and_eval_pytorch.py


In [4]:
# Cell 4: fix a bug in the released code so --gpu_ids works for GPU 1 too, not just GPU 0.
import re

BUGGY_PATTERN = re.compile(r"torch\.device\('cuda:\{\}'\.format\(gpu_ids\[0\]\)\)")
FIXED_SNIPPET = "torch.device('cuda:0') if gpu_ids else torch.device('cpu')"

for script_path in [
    os.path.join(MEDMNIST2D_DIR, "train_and_eval_pytorch.py"),
    os.path.join(MEDMNIST3D_DIR, "train_and_eval_pytorch.py"),
]:
    with open(script_path, "r", encoding="utf-8") as f:
        content = f.read()

    if FIXED_SNIPPET in content:
        print(f"{script_path}: already patched, skipping.")
        continue

    matches = BUGGY_PATTERN.findall(content)
    if len(matches) != 1:
        print(f"{script_path}: WARNING, expected exactly 1 occurrence of the buggy "
              f"line, found {len(matches)}. Not modified; check manually before "
              f"trying to use GPU 1 with this script.")
        continue

    new_content = BUGGY_PATTERN.sub("torch.device('cuda:0')", content, count=1)
    explanation = (
        "# --- PATCHED 2026-09-03 (GPU-selection bugfix, see README) ---\n"
        "# Original code set CUDA_VISIBLE_DEVICES to mask in a single GPU, then addressed\n"
        "# it by its pre-mask index (only correct for GPU 0). After masking to any single\n"
        "# GPU, CUDA always re-numbers it as index 0, so the device line below now always\n"
        "# requests cuda:0; --gpu_ids still correctly controls which physical GPU gets\n"
        "# masked in. Zero effect on model/data/training procedure, only on which\n"
        "# physical chip a run executes on.\n"
    )
    new_content = new_content.replace("def main(", explanation + "def main(", 1)

    with open(script_path, "w", encoding="utf-8") as f:
        f.write(new_content)
    print(f"{script_path}: patched (device-selection line fixed).")

experiments-main\MedMNIST2D\train_and_eval_pytorch.py: already patched, skipping.
experiments-main\MedMNIST3D\train_and_eval_pytorch.py: already patched, skipping.


In [5]:
# Cell 5: dataset configuration, paper target numbers, and the command builder.
# This is the one cell worth reading before hitting "Run All".

NUM_EPOCHS = 100  # paper recipe, used only for the 1% AUC misses below
NUM_RUNS = 15     # PRE-REGISTERED: 15 runs at batch 64. Do not change.
GPU_IDS = [0, 1]  # both RTX 3090s.

RUN_TAG = "ep100_bs64"   # keeps these runs separate from the batch-128 runs

def make_run_name(run_idx):
    return f"run{run_idx}_{RUN_TAG}" if RUN_TAG else f"run{run_idx}"

# Table 3 (ResNet-50, 224px) and Table 4 (ResNet-50+3D) from the MedMNIST v2 paper.
PAPER_RESULTS = {
    # 2D
    "pathmnist":      {"AUC": 0.989, "ACC": 0.892},
    "chestmnist":     {"AUC": 0.773, "ACC": 0.948},
    "dermamnist":     {"AUC": 0.912, "ACC": 0.731},
    "octmnist":       {"AUC": 0.958, "ACC": 0.776},
    "pneumoniamnist": {"AUC": 0.962, "ACC": 0.884},
    "retinamnist":    {"AUC": 0.716, "ACC": 0.511},
    "breastmnist":    {"AUC": 0.866, "ACC": 0.842},
    "bloodmnist":     {"AUC": 0.997, "ACC": 0.950},
    "tissuemnist":    {"AUC": 0.932, "ACC": 0.680},
    "organamnist":    {"AUC": 0.998, "ACC": 0.947},
    "organcmnist":    {"AUC": 0.993, "ACC": 0.911},
    "organsmnist":    {"AUC": 0.975, "ACC": 0.785},
    # 3D
    "organmnist3d":     {"AUC": 0.994, "ACC": 0.883},
    "nodulemnist3d":    {"AUC": 0.875, "ACC": 0.847},
    "fracturemnist3d":  {"AUC": 0.725, "ACC": 0.494},
    "adrenalmnist3d":   {"AUC": 0.828, "ACC": 0.745},
    "vesselmnist3d":    {"AUC": 0.907, "ACC": 0.918},
    "synapsemnist3d":   {"AUC": 0.851, "ACC": 0.795},
}

# PRE-REGISTERED: BreastMNIST only.
DATASETS_2D = [
    "breastmnist",
]
DATASETS_3D = []
ALL_DATASETS = DATASETS_2D + DATASETS_3D

# Only these two 3D datasets use --shape_transform (paper's regularization for
# "shape modality" datasets). Everything else omits it.
SHAPE_TRANSFORM_DATASETS = {"adrenalmnist3d", "vesselmnist3d"}


def build_command(data_flag, run_name, physical_gpu_id):
    """Returns (working_directory, command_list) for one training run."""
    if data_flag in DATASETS_3D:
        cmd = [
            "python", "train_and_eval_pytorch.py",
            "--data_flag", data_flag,
            "--model_flag", "resnet50",
            "--conv", "Conv3d",          # for "ResNet-50+3D" specifically
            "--pretrained_3d", "i3d",
            "--as_rgb",                  # paper: "copy the single channel into 3 channels"
            "--num_epochs", str(NUM_EPOCHS),
            "--batch_size", "32",        # paper's setting for 3D
            "--download",
            "--gpu_ids", str(physical_gpu_id),
            "--run", run_name,
            # Unique tree per run so two jobs starting in the same wall-clock
            # second cannot share the authors' "%y%m%d_%H%M%S" folder. On
            # Windows that collision is FileExistsError WinError 183; using
            # this supported --output_root flag, not a training-recipe change.
            "--output_root", os.path.join("output", run_name),
        ]
        if data_flag in SHAPE_TRANSFORM_DATASETS:
            cmd.append("--shape_transform")
        return MEDMNIST3D_DIR, cmd
    else:
        cmd = [
            "python", "train_and_eval_pytorch.py",
            "--data_flag", data_flag,
            "--model_flag", "resnet50",
            "--size", "28",
            "--resize",                  # reproduces the paper's "224" column
            "--as_rgb",                  # required for the resize path's torchvision ResNet
            "--num_epochs", str(NUM_EPOCHS),
            "--batch_size", "64",        # authors' released R50-224 checkpoints imply 64 (paper text says 128)
            "--download",
            "--gpu_ids", str(physical_gpu_id),
            "--run", run_name,
            "--output_root", os.path.join("output", run_name),
        ]
        return MEDMNIST2D_DIR, cmd


def find_result_csvs(data_flag, run_name):
    """Return existing test-set result CSVs for one (dataset, run), or [].

    Two layouts, both with a single * (Windows-safe; ** glob is unreliable
    on Windows and previously made this skip-check return nothing, which
    started a full 2D retrain):
      output/<dataset>/<timestamp>/...@runN.csv     (original 2D layout)
      output/<run>/<dataset>/<timestamp>/...@runN.csv  (3D unique-per-run)
    """
    import glob
    work_dir = MEDMNIST3D_DIR if data_flag in DATASETS_3D else MEDMNIST2D_DIR
    output_root = os.path.join(work_dir, "output")
    filename = f"{data_flag}_test_*@{run_name}.csv"
    matches = glob.glob(os.path.join(output_root, data_flag, "*", filename))
    matches += glob.glob(os.path.join(output_root, run_name, data_flag, "*", filename))
    return matches


print(f"Configured: {len(ALL_DATASETS)} datasets x {NUM_RUNS} runs = "
      f"{len(ALL_DATASETS) * NUM_RUNS} total jobs, NUM_EPOCHS={NUM_EPOCHS}, "
      f"RUN_TAG={RUN_TAG!r}, datasets={ALL_DATASETS}, GPUs={GPU_IDS}")

Configured: 1 datasets x 15 runs = 15 total jobs, NUM_EPOCHS=100, RUN_TAG='ep100_bs64', datasets=['breastmnist'], GPUs=[0, 1]


In [6]:
# Cell 6: THE MAIN LOOP. Runs every (dataset, run) job that isn't already done,
# 2 at a time (one per GPU). This is the cell that actually takes hours; once
# started, everything else is automatic. Safe to stop and re-run: it always
# checks for existing results first and skips them.
import glob
import os
import queue
import subprocess
import threading
import time
from concurrent.futures import ThreadPoolExecutor

def output_glob(data_flag, run_name):
    """Re-bound here so this cell is safe to re-run after an interrupt
    even if Cell 5 was not re-executed in this kernel session."""
    work_dir = MEDMNIST3D_DIR if data_flag in DATASETS_3D else MEDMNIST2D_DIR
    return os.path.join(work_dir, "output", "**", f"{data_flag}_test_*@{run_name}.csv")

gpu_pool = queue.Queue()
for g in GPU_IDS:
    gpu_pool.put(g)

jobs = [(data_flag, run_idx) for data_flag in ALL_DATASETS for run_idx in range(1, NUM_RUNS + 1)]
_todo = sum(1 for d, i in jobs if not glob.glob(output_glob(d, make_run_name(i)), recursive=True))
print(f"Total jobs (datasets x runs): {len(jobs)}  ->  to train: {_todo}, already done: {len(jobs) - _todo}")

_lock = threading.Lock()
_completed = 0
_skipped = 0
_failed = 0

def run_job(job):
    global _completed, _skipped, _failed
    data_flag, run_idx = job
    run_name = make_run_name(run_idx)

    existing = glob.glob(output_glob(data_flag, run_name), recursive=True)
    if existing:
        print(f"[{data_flag} {run_name}] already done, skipping.")
        with _lock:
            _skipped += 1
        return

    gpu_id = gpu_pool.get()
    try:
        work_dir, cmd = build_command(data_flag, run_name, gpu_id)
        if "--output_root" not in cmd:
            cmd.extend(["--output_root", os.path.join("output", run_name)])
        print(f"[{data_flag} {run_name}] starting on GPU {gpu_id}...")
        t0 = time.time()
        # capture_output so a failure's real error (e.g. a missing package's
        # traceback) is visible here instead of just an exit code, learned this
        # the hard way when all 3D jobs failed with no visible reason at first.
        result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)
        elapsed = time.time() - t0
        if result.returncode == 0:
            print(f"[{data_flag} {run_name}] done on GPU {gpu_id} in {elapsed/60:.1f} min.")
            with _lock:
                _completed += 1
        else:
            print(f"[{data_flag} {run_name}] FAILED on GPU {gpu_id} (exit "
                  f"{result.returncode}) after {elapsed/60:.1f} min. Last part of "
                  f"its error output:")
            print(result.stderr[-1500:] if result.stderr else "(no stderr captured)")
            print(f"[{data_flag} {run_name}] continuing with the remaining jobs regardless.")
            with _lock:
                _failed += 1
    finally:
        gpu_pool.put(gpu_id)

t_start = time.time()
with ThreadPoolExecutor(max_workers=len(GPU_IDS)) as pool:
    list(pool.map(run_job, jobs))
t_total = time.time() - t_start

print(f"\n=== This pass finished in {t_total/3600:.2f} hours ===")
print(f"Newly completed: {_completed}, already done: {_skipped}, failed: {_failed}, total jobs: {len(jobs)}")
if _failed > 0:
    print("Some jobs failed, scroll up for the specific dataset/run and error. "
          "Re-running this cell will retry only the ones that never produced output.")

Total jobs (datasets x runs): 15  ->  to train: 15, already done: 0
[breastmnist run1_ep100_bs64] starting on GPU 0...
[breastmnist run2_ep100_bs64] starting on GPU 1...
[breastmnist run2_ep100_bs64] done on GPU 1 in 3.8 min.
[breastmnist run3_ep100_bs64] starting on GPU 1...
[breastmnist run1_ep100_bs64] done on GPU 0 in 4.0 min.
[breastmnist run4_ep100_bs64] starting on GPU 0...
[breastmnist run3_ep100_bs64] done on GPU 1 in 3.8 min.
[breastmnist run5_ep100_bs64] starting on GPU 1...
[breastmnist run4_ep100_bs64] done on GPU 0 in 4.1 min.
[breastmnist run6_ep100_bs64] starting on GPU 0...
[breastmnist run5_ep100_bs64] done on GPU 1 in 3.9 min.
[breastmnist run7_ep100_bs64] starting on GPU 1...
[breastmnist run6_ep100_bs64] done on GPU 0 in 4.2 min.
[breastmnist run8_ep100_bs64] starting on GPU 0...
[breastmnist run7_ep100_bs64] done on GPU 1 in 3.8 min.
[breastmnist run9_ep100_bs64] starting on GPU 1...
[breastmnist run8_ep100_bs64] done on GPU 0 in 4.2 min.
[breastmnist run10_ep100_

In [7]:
# Cell 7: batch 64 vs batch 128 vs the authors' own 3 runs. Safe to run at any time.
import csv
import re

import numpy as np
from scipy import stats

PAPER = PAPER_RESULTS["breastmnist"]["AUC"]
AUTHORS_RUNS = [0.839, 0.906, 0.853]  # Zenodo 7782114 predictions, breastmnist resnet50_224 runs 1-3


def collect(run_names):
    out = []
    for run_name in run_names:
        matches = sorted(find_result_csvs("breastmnist", run_name))
        if not matches:
            continue
        if len(matches) > 1:
            print(f"WARNING {run_name}: {len(matches)} result files, using the newest: {matches[-1]}")
        m = re.search(r"\[AUC\]([\d.]+)_\[ACC\]([\d.]+)", matches[-1])
        out.append({"run": run_name, "test_auc": float(m.group(1)), "test_acc": float(m.group(2)),
                    "file": matches[-1]})
    return out


bs64 = collect([make_run_name(i) for i in range(1, NUM_RUNS + 1)])
bs128 = collect([f"run{i}_ep100" for i in range(1, 16)])


def describe(label, aucs):
    a = np.array(aucs)
    n, mean, sd = len(a), a.mean(), a.std(ddof=1)
    half = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
    return {"setting": label, "n_runs": n, "mean": round(mean, 4), "sd": round(sd, 4),
            "ci95_low": round(mean - half, 4), "ci95_high": round(mean + half, 4),
            "gap_pts_vs_paper": round((mean - PAPER) * 100, 2),
            "paper_inside_ci": bool(mean - half <= PAPER <= mean + half),
            "min": a.min(), "max": a.max()}


print("batch 64 :", ", ".join(f"{r['test_auc']:.3f}" for r in bs64))
print("batch 128:", ", ".join(f"{r['test_auc']:.3f}" for r in bs128))
print("authors  :", ", ".join(f"{x:.3f}" for x in AUTHORS_RUNS), f"(mean {np.mean(AUTHORS_RUNS):.4f} = paper {PAPER})")

rows = []
if len(bs64) >= 2:
    rows.append(describe("batch 64", [r["test_auc"] for r in bs64]))
if len(bs128) >= 2:
    rows.append(describe("batch 128", [r["test_auc"] for r in bs128]))
print()
print(f"{'setting':<11}{'runs':>5}{'mean':>8}{'sd':>8}{'95% CI':>18}{'gap(pts)':>10}  paper in CI")
for r in rows:
    print(f"{r['setting']:<11}{r['n_runs']:>5}{r['mean']:>8.4f}{r['sd']:>8.4f}   [{r['ci95_low']:.3f}, {r['ci95_high']:.3f}]"
          f"{r['gap_pts_vs_paper']:>+10.2f}  {r['paper_inside_ci']}")
if len(bs64) >= 2 and len(bs128) >= 2:
    t = stats.ttest_ind([r["test_auc"] for r in bs64], [r["test_auc"] for r in bs128], equal_var=False)
    print(f"\nbatch 64 vs batch 128, Welch t-test p = {t.pvalue:.3f}")
    t2 = stats.ttest_ind([r["test_auc"] for r in bs64], AUTHORS_RUNS, equal_var=False)
    print(f"batch 64 vs the authors' 3 runs, Welch t-test p = {t2.pvalue:.3f}")

with open("bs64_per_run.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["run", "test_auc", "test_acc", "file"])
    w.writeheader()
    w.writerows(bs64)
if rows:
    with open("bs64_summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
print("\nSaved bs64_per_run.csv and bs64_summary.csv")
print("ALL 15 BATCH-64 RUNS COMPLETE" if len(bs64) == NUM_RUNS else
      f"Only {len(bs64)}/{NUM_RUNS} batch-64 runs found: re-run Cell 6 (finished runs are skipped), then this cell.")


batch 64 : 0.837, 0.902, 0.881, 0.862, 0.867, 0.856, 0.854, 0.826, 0.809, 0.873, 0.864, 0.823, 0.889, 0.877, 0.842
batch 128: 0.787, 0.854, 0.832, 0.805, 0.852, 0.858, 0.851, 0.871, 0.832, 0.841, 0.851, 0.859, 0.887, 0.792, 0.772
authors  : 0.839, 0.906, 0.853 (mean 0.8660 = paper 0.866)

setting     runs    mean      sd            95% CI  gap(pts)  paper in CI
batch 64      15  0.8575  0.0261   [0.843, 0.872]     -0.85  True
batch 128     15  0.8363  0.0331   [0.818, 0.855]     -2.97  False

batch 64 vs batch 128, Welch t-test p = 0.062
batch 64 vs the authors' 3 runs, Welch t-test p = 0.723

Saved bs64_per_run.csv and bs64_summary.csv
ALL 15 BATCH-64 RUNS COMPLETE
